# **Simple Gen AI Application**

In [ ]:
!pip install -U langchain langchain-community langchain-core huggingface_hub faiss-cpu


Environment loading

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"]='true'
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_Token")

In [ ]:
# LLM + embeddings + vectorstore
from langchain_community.llms import HuggingFaceHub
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


Data Ingestion

In [ ]:
#data ingestion from the website we need to scrap the data
from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader('https://docs.langchain.com/langsmith/home')
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployme

Data Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
documents=text_splitter.split_documents(docs)


In [ ]:
documents[0]

Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly')

Text Embedding

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
Database=FAISS.from_documents(documents ,embeddings)

/tmp/ipython-input-2717753753.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#querying with the database
query="What are the tools provided by langSmith ?"
result=Database.similarity_search_with_score(query)
result[0]

(Document(id='f57da1d7-d832-4f9e-8ae1-0ee5af63541c', metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly'),
 np.float32(0.9246493))

# **Retrieval Chain- QA**

**Convert Vector Store → Retriever**

In [ ]:
retriever = Database.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)


**HuggingFaceHub LLM**

In [ ]:

llm = HuggingFaceHub(
    repo_id="google/flan-t5-base",
    task="text2text-generation",
    model_kwargs={
        "temperature": 0.2,
        "max_length": 512
    }
)



**Prompt + chains**

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

<context>
{context}
</context>

Question: {input}
""")



**Retrieval + generation chain**

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retrieval_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


**Querying**

In [ ]:
response = retrieval_chain.invoke("What tools does LangSmith provide?")
print(response)


AttributeError: 'InferenceClient' object has no attribute 'post'